In [2]:
import sqlite3
import pandas as pd

# Crear conexión a la base de datos
conn = sqlite3.connect('../database/erillam.db')
cursor = conn.cursor()

# ─────────────────────────────────────────
# TABLA 1: productos
# Catálogo maestro de insumos médicos
# ─────────────────────────────────────────
cursor.execute('''
CREATE TABLE IF NOT EXISTS productos (
    id_producto         INTEGER PRIMARY KEY AUTOINCREMENT,
    clave_insumo        TEXT UNIQUE NOT NULL,
    descripcion         TEXT NOT NULL,
    tipo_insumo         TEXT,  -- medicamento / material de curacion
    grupo_terapeutico   TEXT,
    activo              INTEGER DEFAULT 1
)''')

# ─────────────────────────────────────────
# TABLA 2: inventario_diario
# Snapshot diario del stock real del ISSSTE
# ─────────────────────────────────────────
cursor.execute('''
CREATE TABLE IF NOT EXISTS inventario_diario (
    id_registro             INTEGER PRIMARY KEY AUTOINCREMENT,
    id_producto             INTEGER NOT NULL,
    fecha_corte             TEXT NOT NULL,
    inventario_piezas       INTEGER,
    demanda_mensual_nacional INTEGER,
    dias_cobertura          REAL,  -- calculado: inventario / (demanda/30)
    estatus_stock           TEXT,  -- critico / bajo / normal / optimo
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
)''')

# ─────────────────────────────────────────
# TABLA 3: alertas_desabasto
# Productos en riesgo identificados por el ETL
# ─────────────────────────────────────────
cursor.execute('''
CREATE TABLE IF NOT EXISTS alertas_desabasto (
    id_alerta       INTEGER PRIMARY KEY AUTOINCREMENT,
    id_producto     INTEGER NOT NULL,
    fecha_alerta    TEXT NOT NULL,
    tipo_alerta     TEXT,  -- critico / preventivo
    dias_cobertura  REAL,
    mensaje         TEXT,
    activa          INTEGER DEFAULT 1,
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
)''')

# ─────────────────────────────────────────
# TABLA 4: resumen_grupos
# Agregado por grupo terapéutico por día
# ─────────────────────────────────────────
cursor.execute('''
CREATE TABLE IF NOT EXISTS resumen_grupos (
    id_resumen              INTEGER PRIMARY KEY AUTOINCREMENT,
    grupo_terapeutico       TEXT NOT NULL,
    fecha_corte             TEXT NOT NULL,
    total_productos         INTEGER,
    productos_criticos      INTEGER,
    productos_bajo_stock    INTEGER,
    inventario_total        INTEGER,
    demanda_total           INTEGER
)''')

conn.commit()
conn.close()
print("✅ Base de datos creada correctamente con 4 tablas")
print("   → productos")
print("   → inventario_diario")
print("   → alertas_desabasto")
print("   → resumen_grupos")


✅ Base de datos creada correctamente con 4 tablas
   → productos
   → inventario_diario
   → alertas_desabasto
   → resumen_grupos
